In [1]:
!pip install transformers datasets -q
!pip install transformers -q
!pip install keras_nlp -q
!pip install datasets -q
!pip install huggingface-hub -q
!pip install nltk -q
!pip install rouge-score -q
!pip install huggingface_hub
!pip install rouge-score -q
!pip install datasets -q
!pip install evaluate -q
!pip install bert-score -q

import torch
from transformers import PegasusForConditionalGeneration, PegasusTokenizer, DataCollatorForSeq2Seq
from transformers import AdamW
from datasets import Dataset
from transformers import get_scheduler
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd
import numpy as np
from rouge_score import rouge_scorer
from sklearn.metrics import average_precision_score
import gc
import string
from string import punctuation
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from torch.cuda.amp import autocast, GradScaler
from transformers import PegasusConfig

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.3 MB/s eta 0:00:00
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
class PegasusMemoryConfig(PegasusConfig):
    def __init__(self, n_gram_size=3, max_memory_size=10000, **kwargs):
        super().__init__(**kwargs)
        self.n_gram_size = n_gram_size
        self.max_memory_size = max_memory_size

class PegasusWithMemory(PegasusForConditionalGeneration):
    @classmethod
    def from_pretrained(cls, pretrained_model_name_or_path, *model_args, **kwargs):
        # First load the base model configuration
        config = kwargs.pop('config', None)
        if config is None:
            config = PegasusMemoryConfig.from_pretrained(pretrained_model_name_or_path)
            
        # Ensure memory-specific parameters are set
        if not hasattr(config, 'n_gram_size'):
            config.n_gram_size = 3
        if not hasattr(config, 'max_memory_size'):
            config.max_memory_size = 10000

        # Initialize model with the configuration
        model = super().from_pretrained(
            pretrained_model_name_or_path,
            config=config,
            *model_args,
            **kwargs
        )
        
        # Initialize memory-specific attributes
        model.memory = {}
        model.tokenizer = PegasusTokenizer.from_pretrained(pretrained_model_name_or_path)
        
        return model

    #N-Grams

    def remove_punctuation(self, text):
        if(type(text)==float):
            return text
        ans=""  
        for i in text:     
            if i not in string.punctuation:
                ans+=i    
        return ans

    def generate_n_grams(self, text, n = 3):
        text = self.remove_punctuation(text)
        words = [word for word in text.split(" ") if word not in set(stopwords.words('english'))]
        n_grams = zip(*[words[i:] for i in range(n)])
        return [' '.join(ngram) for ngram in n_grams]

    def update_memory(self, text, input_data):
        ngrams = self.generate_n_grams(text)
        # input_data = self.remove_punctuation(input_data)
        # data = ""
        # for word in input_data.split(" "):
        #     if word not in set(stopwords.words('english')):
        #         data += " "+word
        for ngram in ngrams:
            if ngram not in self.memory:
                self.memory[ngram] = set()
            self.memory[ngram].add(input_data)
            
            if len(self.memory[ngram]) > self.config.max_memory_size:
                list(self.memory[ngram]).pop(0)

    #TF-IDF

    def prepare_retrieval_data(self, corpus, query, top_k=2, threshold=0.6):
        if isinstance(corpus[0], list):
            corpus = [" ".join(doc) for doc in corpus]
        
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(corpus + [query])
        query_vector = tfidf_matrix[-1]

        # similarities = cosine_similarity(query_vector, tfidf_matrix[:-1])
        # top_k_indices = similarities.argsort()[0][-top_k:][::-1]
        # return [corpus[idx] for idx in top_k_indices]

        # Compute cosine similarities between the query and all corpus documents.
        # This returns a 2D array with shape (1, num_docs).
        # Flatten the result so that each similarity is a float.
        similarities = cosine_similarity(query_vector, tfidf_matrix[:-1]).flatten()
    
        # Filter for indices where the similarity is above the threshold.
        valid_indices = [i for i, sim in enumerate(similarities) if sim > threshold]
        
        # If no document meets the threshold, return an empty list.
        if not valid_indices:
            return []
        
        # Sort the valid indices in descending order of similarity.
        valid_indices = sorted(valid_indices, key=lambda i: similarities[i], reverse=True)
        
        # Select the top_k indices from the sorted valid indices.
        top_k_indices = valid_indices[:top_k]
        
        # Return the corresponding documents from the corpus.
        return [corpus[idx] for idx in top_k_indices]

    def retrieve_memory(self, text):
        ngrams = self.generate_n_grams(text)
        relevant_data = []
        output = []
        for ngram in ngrams:
            if ngram in self.memory:
                relevant_data.extend(self.memory[ngram])
                output = self.prepare_retrieval_data(list(set(relevant_data)), text)
                
        return output

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        if input_ids is not None:
            # Get original text
            text = self.tokenizer.batch_decode(input_ids, skip_special_tokens=True)[0]
            memory_data = self.retrieve_memory(text)
            # # print(f"Retrieved Info: {memory_data}")

            # if memory_data:
            #     # Augment input with memory
            #     # augmented_text = " ".join(memory_data) + " " + text
            #     # augmented_text = "(" + memory_data[0] + ")" + "\n" + text
                
            #     augmented_text = memory_data[0] + " " + text
            #     # augmented_text = text + " " + memory_data[0]
                
            #     # print(f"Augmented Text: {augmented_text}")
            #     # augmented_text = "Additional Info: " + memory_data[0] + ", Summarize: " + text
            #     inputs = self.tokenizer(
            #         augmented_text,
            #         truncation=True,
            #         padding=True,
            #         max_length=self.config.max_position_embeddings,
            #         return_tensors="pt"
            #     ).to(input_ids.device)

            #     input_ids = inputs["input_ids"]
            #     attention_mask = inputs["attention_mask"]

            # Update memory
            
            # input_data = self.remove_punctuation(text)
            # data = " ".join([word for word in input_data.split(" ") if word not in set(stopwords.words('english'))][:25])
            
            # data = text[:150]
            # data = text
            # data = self.remove_punctuation(data)
            # data = " ".join([word for word in data.split(" ") if word not in set(stopwords.words('english'))])
            
            # data = ""
            # counter = 0
            # for word in text.split(" "):
            #     if counter < 10:
            #         if word not in set(stopwords.words('english')) and word != "":
            #             data += word + " "
            #             counter += 1
            #     else:
            #         break

            if labels != None:
                data = ""
                 # Loop through each sequence in the batch
                for label_seq in labels:
                    # Convert the tensor to a list of token IDs
                    token_ids = label_seq.tolist()
                    
                    # Filter out the ignore index (-100)
                    filtered_ids = [token for token in token_ids if token != -100]
                    
                    # Decode the filtered token IDs to text
                    decoded_text = tokenizer.decode(filtered_ids, skip_special_tokens=True)
                    data += decoded_text
        
                # print("Decoded Label:", text)
                
                self.update_memory(text, data)

        output = super().forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            **kwargs
        )

        # summary = self.tokenizer.decode(labels, skip_special_tokens=True)
        # print(f"Label: {labels}")
            
        # predicted_ids = output.logits.argmax(dim=-1)
        # text = self.tokenizer.decode(predicted_ids[0], skip_special_tokens=True)
        # # decoded_texts = [self.tokenizer.decode(ids, skip_special_tokens=True) for ids in predicted_ids]
        # print(f"Output: {text}")
        return output

    def generate(self, input_ids=None, attention_mask=None, **kwargs):
        if input_ids is not None:
            text = self.tokenizer.batch_decode(input_ids, skip_special_tokens=True)[0]
            memory_data = self.retrieve_memory(text)
            # print(f"Retrieved Info: {memory_data}")

            if memory_data:
                # augmented_text = memory_data[0] + "\n" + text
                augmented_text = text + " " + memory_data[0]
                
                # print(f"Augmented Text: {augmented_text}")
                
                inputs = self.tokenizer(
                    augmented_text,
                    truncation=True,
                    padding=True,
                    max_length=self.config.max_position_embeddings,
                    return_tensors="pt"
                ).to(input_ids.device)

                input_ids = inputs["input_ids"]
                attention_mask = inputs["attention_mask"]

            # Update memory
            # input_data = self.remove_punctuation(text)
            # data = " ".join([word for word in input_data.split(" ") if word not in set(stopwords.words('english'))][:25])

            
            # data = text[:150]
            
            # data = text

            # data = ""
            # counter = 0
            # for word in text.split(" "):
            #     if counter < 10:
            #         if word not in set(stopwords.words('english')) and word != "":
            #             data += word + " "
            #             counter += 1
            #     else:
            #         break
            
            # self.update_memory(text, data)

        output = super().generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs
        )

        data = self.tokenizer.decode(output[0], skip_special_tokens=True)
        # print(f"Generated Output: {text}")
        self.update_memory(text, data)
        
        return output

# Initialize the model using the from_pretrained method
# model = PegasusWithMemory.from_pretrained("google/pegasus-cnn_dailymail")
# tokenizer = PegasusTokenizer.from_pretrained('google/pegasus-cnn_dailymail')

model = PegasusWithMemory.from_pretrained("google/pegasus-xsum")
base_model = PegasusForConditionalGeneration.from_pretrained("google/pegasus-xsum")
tokenizer = PegasusTokenizer.from_pretrained('google/pegasus-xsum')

# model = PegasusWithMemory.from_pretrained("google/pegasus-x-base")
# base_model = PegasusForConditionalGeneration.from_pretrained("google/pegasus-x-base")
# tokenizer = PegasusTokenizer.from_pretrained('google/pegasus-x-base')

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
base_model = base_model.to(device)

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusWithMemory were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.52M [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
from torch.optim import AdamW
from torch.optim import Adafactor
from nltk.translate.bleu_score import sentence_bleu
from sklearn.metrics import average_precision_score
from datasets import load_dataset

import pandas as pd
from datasets import Dataset

# CNN-DailyNews
# data = load_dataset("cnn_dailymail", "3.0.0")
# data = load_dataset("LogeshChandran/newsroom", trust_remote_code=True)
# data = load_dataset("xsum", trust_remote_code=True)

# train_df = data["train"]
# val_df = data["validation"]
# test_df = data["test"]

# # sub_dataset = train_df.train_test_split(train_size=0.005, seed=42)
# train_dataset = train_df.shuffle(seed=42).select(range(1000))
# val_dataset = val_df.shuffle(seed=42).select(range(200))
# test_dataset = test_df.shuffle(seed=42).select(range(200))

# print(train_dataset)
# print(val_dataset)
# print(test_dataset)

# Load CSV file
train_csv_path = "/kaggle/input/indisumm/IndiSummDataset_Train.csv"
validation_csv_path = "/kaggle/input/indisumm/IndiSummDataset_Validation.csv"
test_csv_path = "/kaggle/input/indisumm/IndiSummDataset_Test.csv"

train_data = pd.read_csv(train_csv_path)
valid_data = pd.read_csv(validation_csv_path)
test_data = pd.read_csv(test_csv_path)

train_df = Dataset.from_pandas(train_data)
val_df = Dataset.from_pandas(valid_data)
test_df = Dataset.from_pandas(test_data)

train_dataset = train_df
val_dataset = val_df
test_dataset = test_df

print(f"\nTrain: {train_dataset}\n")
print(f"\nVal: {val_dataset}\n")
print(f"\nTest: {test_dataset}\n")


Train: Dataset({
    features: ['Headline', 'Content', 'Category', 'Human Summary'],
    num_rows: 241
})


Val: Dataset({
    features: ['Headline', 'Content', 'Category', 'Human Summary'],
    num_rows: 50
})


Test: Dataset({
    features: ['Headline', 'Content', 'Category', 'Human Summary'],
    num_rows: 74
})



In [4]:
import torch
from torch.utils.data import DataLoader
from transformers import get_scheduler

import torch
from torch.utils.data import DataLoader
from transformers import get_scheduler

# def train_model_on_csv(train_dataset, val_dataset, model, tokenizer, optimizer, device, epochs=1, batch_size=1, grad_accum_steps=4):
#     model.to(device)
#     model.train()
    
#     train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
#     val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

#     num_training_steps = epochs * len(train_dataloader) // grad_accum_steps  # Adjust for grad accumulation
#     lr_scheduler = get_scheduler(
#         "linear",
#         optimizer=optimizer,
#         num_warmup_steps=100,
#         num_training_steps=num_training_steps
#     )

#     # scaler = torch.amp.GradScaler(device_type="cuda")  # Fixed deprecation warning
#     # scaler = torch.cuda.amp.GradScaler()
#     scaler = torch.amp.GradScaler('cuda')

#     for epoch in range(epochs):
#         total_loss = 0
#         model.train()

#         for step, batch in enumerate(train_dataloader):
#             context = batch["article"]
#             reference = batch["highlights"]

#             inputs = tokenizer(context, truncation=True, padding="max_length", return_tensors="pt")
#             labels = tokenizer(reference, truncation=True, padding="max_length", return_tensors="pt").input_ids

#             inputs, labels = inputs.to(device), labels.to(device)
#             labels[labels == tokenizer.pad_token_id] = -100  # Ignore padding tokens

#             optimizer.zero_grad(set_to_none=True)  # Free memory efficiently

#             with torch.amp.autocast("cuda"):  # Fixed deprecation warning
#                 outputs = model(**inputs, labels=labels)
#                 loss = outputs.loss / grad_accum_steps  # Normalize loss for accumulation

#             scaler.scale(loss).backward()

#             if (step + 1) % grad_accum_steps == 0:  # Only update every `grad_accum_steps`
#                 torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#                 scaler.step(optimizer)
#                 scaler.update()
#                 lr_scheduler.step()
#                 optimizer.zero_grad(set_to_none=True)

#             total_loss += loss.item() * grad_accum_steps  # Reverse normalization

#             # Free memory
#             del inputs, labels, outputs, loss
#             torch.cuda.empty_cache()

#         avg_train_loss = total_loss / len(train_dataloader)
#         print(f"Epoch {epoch+1} Training Loss = {avg_train_loss:.4f}")

#         # Validate model after each epoch
#         val_loss = evaluate_model(val_dataloader, model, tokenizer, device)
#         print(f"Epoch {epoch+1} Validation Loss = {val_loss:.4f}") 
    
#     print("Training Completed")

# def evaluate_model(val_dataloader, model, tokenizer, device):
#     model.eval()
#     total_loss = 0

#     with torch.no_grad():
#         for batch in val_dataloader:
#             context = batch["article"]
#             reference = batch["highlights"]

#             inputs = tokenizer(context, truncation=True, padding="max_length", return_tensors="pt")
#             labels = tokenizer(reference, truncation=True, padding="max_length", return_tensors="pt").input_ids

#             inputs, labels = inputs.to(device), labels.to(device)
#             labels[labels == tokenizer.pad_token_id] = -100  

#             with torch.amp.autocast("cuda"):
#                 outputs = model(**inputs, labels=labels)
#                 loss = outputs.loss

#             total_loss += loss.item()

#             del inputs, labels, outputs, loss
#             torch.cuda.empty_cache()

#     return total_loss / len(val_dataloader)



def train_model_on_csv(train_dataset, val_dataset, model, tokenizer, optimizer, epochs=1):
    model.to(device)
    model.train()
    
    num_training_steps = epochs * len(train_dataset)
    
    # Learning rate scheduler
    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=10,
        num_training_steps=num_training_steps
    )

    # prev_val_loss = -1
    # epoch = 0
    for epoch in range(epochs):
    # while(True):
        epoch_loss = 0
        for batch in train_dataset:
            # Others
            # context = batch["document"]
            # reference = batch["summary"]

            # Newsroom
            # context = batch["text"]
            # reference = batch["summary"]
            
            # CNN-DailyMail News

            # context = batch["article"]
            # reference = batch["highlights"]

            
            # Custom CSV Data

            category = batch["Category"]
            headline = batch["Headline"]
            context = batch["Content"]
            reference = batch["Human Summary"]
            
            # Prepare input text
            input_text = f"{category}\n{headline}\n{context}"
            
            # Tokenize inputs and targets
            # inputs = tokenizer(context, truncation=True, padding="max_length", return_tensors="pt").to(device)
            inputs = tokenizer(input_text, truncation=True, padding="max_length", return_tensors="pt").to(device)
            labels = tokenizer(reference, truncation=True, padding="max_length", return_tensors="pt").input_ids.to(device)
            
            # Handle padding tokens
            labels[labels == tokenizer.pad_token_id] = -100
            
            # Zero gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss
            
            # Backpropagation
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Optimizer and scheduler steps
            optimizer.step()
            lr_scheduler.step()
            
            epoch_loss += loss.item()
            
        avg_epoch_loss = epoch_loss / len(train_dataset)
        print(f"Epoch {epoch+1} Training Loss = {avg_epoch_loss:.4f}")

        # Validate model after each epoch
        val_loss = evaluate_model(val_dataset, model, tokenizer, device)
        print(f"Epoch {epoch+1} Validation Loss = {val_loss:.4f}")

        # calculate_metrics_on_csv(val_dataset, model, tokenizer)

        # if prev_val_loss != -1 and prev_val_loss <= val_loss:
        #     break

        # prev_val_loss = val_loss
        # epoch += 1

    
    print("Training Completed")

def evaluate_model(val_dataset, model, tokenizer, device="cuda"):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in val_dataset:
            # Others
            # context = batch["document"]
            # reference = batch["summary"]
            
            # Newsroom
            # context = batch["text"]
            # reference = batch["summary"]
            
            # CNN-DailyMail News

            # context = batch["article"]
            # reference = batch["highlights"]            
            
            # Custom CSV Data
            
            category = batch["Category"]
            headline = batch["Headline"]
            context = batch["Content"]
            reference = batch["Human Summary"]

            # Prepare input text
            input_text = f"{category}\n{headline}\n{context}"

            # Tokenize inputs and targets
            # inputs = tokenizer(context, truncation=True, padding="max_length", return_tensors="pt").to(device)
            inputs = tokenizer(input_text, truncation=True, padding="max_length", return_tensors="pt").to(device)
            labels = tokenizer(reference, truncation=True, padding="max_length", return_tensors="pt").input_ids.to(device)

            # Handle padding tokens
            labels[labels == tokenizer.pad_token_id] = -100
            
            # Forward pass
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

    avg_val_loss = total_loss / len(val_dataset)
    return avg_val_loss

In [5]:
import torch
from rouge_score import rouge_scorer

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.metrics import precision
from collections import Counter
from bert_score import score as bert_score

def calculate_metrics_on_csv(test_dataset, model, tokenizer, device="cuda"):
    model.eval()
    model.to(device)

    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    predictions = []
    references = []

    with torch.no_grad():
        for batch in test_dataset:  
            # Others
            # context = batch["document"]
            # reference = batch["summary"]

            # Newsroom
            # context = batch["text"]
            # reference = batch["summary"]
            
            # CNN-DailyMail News

            # context = batch["article"]
            # reference = batch["highlights"]

            # Custom CSV Data
            
            category = batch["Category"]
            headline = batch["Headline"]
            context = batch["Content"]
            reference = batch["Human Summary"]

            # Prepare input text
            input_text = f"{category}\n{headline}\n{context}"
            
            # Tokenize inputs and generate summaries
            # inputs = tokenizer(context, truncation=True, padding="max_length", return_tensors="pt").to(device)
            inputs = tokenizer(input_text, truncation=True, padding="max_length", return_tensors="pt").to(device)
            # summary_ids = model.generate(**inputs, max_length=150, num_beams=4)
            summary_ids = model.generate(**inputs)
            generated_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

            # Collect predictions and references
            predictions.append(generated_text)
            references.append(reference)

    rouge_scores = { "rouge1": [], "rouge2": [], "rougeL": [] }
    precision_scores = []
    
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge_scores["rouge1"].append(scores["rouge1"].fmeasure)
        rouge_scores["rouge2"].append(scores["rouge2"].fmeasure)
        rouge_scores["rougeL"].append(scores["rougeL"].fmeasure)

        # **Average Precision Calculation (Word-Level)**
        ref_words = Counter(ref.split())
        pred_words = Counter(pred.split())
        common_words = sum((ref_words & pred_words).values())  # Correct words retrieved
        total_predicted_words = sum(pred_words.values())  # Total words in generated summary
        precision_score = common_words / total_predicted_words if total_predicted_words > 0 else 0
        precision_scores.append(precision_score)

        ref_tokens = [ref.split()]  # BLEU expects a list of reference token lists
        pred_tokens = pred.split()

    avg_rouge_scores = {key: sum(values) / len(values) for key, values in rouge_scores.items()}
    avg_precision = sum(precision_scores) / len(precision_scores)

    # Compute BERT Score over all predictions and references.
    # BERT Score computes precision, recall, and F1; here we use F1 as our semantic similarity metric.
    P, R, F1 = bert_score(predictions, references, lang="en", verbose=True)
    avg_bert_score = F1.mean().item()

    print(f"Average ROUGE Scores: {avg_rouge_scores}")
    print(f"Average Precision: {avg_precision:.4f}")
    print(f"Average BERT Score (F1): {avg_bert_score:.4f}")

In [6]:
# optimizer = AdamW(model.parameters(), lr=5e-5)
optimizer = Adafactor(model.parameters(), lr=5e-5)

In [7]:
# train_loss = train_model_on_csv(train_dataset, val_dataset, model, tokenizer, optimizer, device, epochs=4, batch_size=1, grad_accum_steps=8)
train_loss = train_model_on_csv(train_dataset, val_dataset, model, tokenizer, optimizer, epochs=3)

Epoch 1 Training Loss = 2.1884
Epoch 1 Validation Loss = 1.7094
Epoch 2 Training Loss = 1.4452
Epoch 2 Validation Loss = 1.6083
Epoch 3 Training Loss = 1.2579
Epoch 3 Validation Loss = 1.6041
Training Completed


In [8]:
calculate_metrics_on_csv(test_dataset, model, tokenizer)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 2.12 seconds, 34.91 sentences/sec
Average ROUGE Scores: {'rouge1': 0.4075923728056716, 'rouge2': 0.20210377935668408, 'rougeL': 0.2950427506352136}
Average Precision: 0.4668
Average BERT Score (F1): 0.8870


In [9]:
# optimizer = AdamW(base_model.parameters(), lr=5e-5)
optimizer = Adafactor(base_model.parameters(), lr=5e-5)

In [10]:
# train_loss = train_model_on_csv(train_dataset, val_dataset, base_model, tokenizer, optimizer, device, epochs=4, batch_size=1, grad_accum_steps=8)
train_loss = train_model_on_csv(train_dataset, val_dataset, base_model, tokenizer, optimizer, epochs=3)

Epoch 1 Training Loss = 2.1273
Epoch 1 Validation Loss = 1.6958
Epoch 2 Training Loss = 1.4419
Epoch 2 Validation Loss = 1.6062
Epoch 3 Training Loss = 1.2553
Epoch 3 Validation Loss = 1.6015
Training Completed


In [11]:
calculate_metrics_on_csv(test_dataset, base_model, tokenizer)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 2.10 seconds, 35.16 sentences/sec
Average ROUGE Scores: {'rouge1': 0.41954377597524617, 'rouge2': 0.20814982552354436, 'rougeL': 0.3040794087736646}
Average Precision: 0.4719
Average BERT Score (F1): 0.8870


In [12]:
# text = """India successfully launched Chandrayaan-4, aiming to study the lunar surface and uncover its secrets."""

# inputs = tokenizer(text, truncation=True, padding="max_length", return_tensors="pt").to(device)
# # summary_ids = model.generate(**inputs, max_length=150, num_beams=4)
# summary_ids = model.generate(**inputs)
# generated_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# print(f"Output: {generated_text}")

In [13]:
# text = """Jasvir Singh Garhi, Punjab BSP president,  claims the BSP-SAD alliance will prioritize justice in sacrilege cases, combat corruption, and improve infrastructure if elected.  Key election issues include employment,  drug control, and farmer welfare.  Garhi dismisses concerns about internal party dissent and ticket distribution,  predicting a strong showing, especially in the Doaba region, and refuting  opinion polls favoring AAP or Congress. He criticizes the Congress government's performance and alleges the party is using Dalit leader Channi for political gain."""

# inputs = tokenizer(text, truncation=True, padding="max_length", return_tensors="pt").to(device)
# # summary_ids = model.generate(**inputs, max_length=150, num_beams=4)
# summary_ids = model.generate(**inputs)
# generated_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# print(f"Output: {generated_text}")

In [14]:
# text = """India successfully launched Chandrayaan-4, aiming to study the lunar surface and uncover its secrets."""

# inputs = tokenizer(text, truncation=True, padding="max_length", return_tensors="pt").to(device)
# # summary_ids = model.generate(**inputs, max_length=150, num_beams=4)
# summary_ids = model.generate(**inputs)
# generated_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# print(f"Output: {generated_text}")

In [15]:
# text = """Jasvir Singh Garhi, Punjab BSP president,  claims the BSP-SAD alliance will prioritize justice in sacrilege cases, combat corruption, and improve infrastructure if elected.  Key election issues include employment,  drug control, and farmer welfare.  Garhi dismisses concerns about internal party dissent and ticket distribution,  predicting a strong showing, especially in the Doaba region, and refuting  opinion polls favoring AAP or Congress. He criticizes the Congress government's performance and alleges the party is using Dalit leader Channi for political gain."""

# # inputs = tokenizer(text, truncation=True, padding="max_length", return_tensors="pt").to(device)
# # summary_ids = model.generate(**inputs, max_length=150, num_beams=4)
# # inputs = tokenizer(text, return_tensors="pt", max_length=4096, truncation=True).to(device)
# inputs = tokenizer(text, truncation=True, padding="max_length", return_tensors="pt").to(device)
# summary_ids = model.generate(**inputs)
# generated_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# print(f"Output: {generated_text}")

In [16]:
category = "Politics"
headline = "Delhi new CM announcement live: BJP leaders want chief minister pick to be a first-time MLA"
content = "Delhi new CM announcement live: The BJP secured a decisive victory in Delhi, winning 48 out of 70 assembly seats. AAP claimed 22 seats, while Congress faced another defeat, failing to secure any seats in the February 5 elections. AAP leader Atishi resigned as the chief minister of Delhi on Sunday, clearing the way for the formation of a new government by the BJP after its return to power in the capital after 27 years. The party has yet to announce its chief ministerial candidate. A BJP leader mentioned that the oath-taking ceremony will take place after Prime Minister Narendra Modi returns from his official visit to the US, which is scheduled for later this week. He is expected to be back on February 14 or 15. A day after the BJP secured victory in the Delhi assembly elections, its winning candidates held thanksgiving rallies in their respective constituencies on Sunday. They celebrated their success by distributing sweets, cutting cakes, and expressing gratitude to voters for their support. The BJP returned to power in Delhi with a decisive mandate, winning 48 out of 70 assembly seats on Saturday. AAP secured 22 seats, while Congress suffered another setback, failing to win a single seat in the February 5 elections. This marks the third consecutive time that Congress has drawn a blank in the Delhi assembly elections."

text = f"{category}\n{headline}\n{content}"

inputs = tokenizer(text, truncation=True, padding="max_length", return_tensors="pt").to(device)
summary_ids = model.generate(**inputs)
generated_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f"Output: {generated_text}")

Output: A day after the BJP secured victory in the Delhi assembly elections, its winning candidates held thanksgiving rallies in their respective constituencies. AAP leader Atishi resigned as the chief minister of Delhi on Sunday, clearing the way for the formation of a new government by the BJP after its return to power in the capital after 27 years
